# Weeks Segmented

Splits the H&M transactions into one CSV per week, so the pipeline can ingest them one week at a time (notebook 02).

1. Load the transactions and parse the dates
2. Cutoff = one month before the last transaction (2020-09-22 → 2020-08-22)
3. Write every transaction after the cutoff to one file per ISO week (Monday to Sunday), named `week_<nn>_<first date>_to_<last date>.csv`

The master table (`hnm_transactions.csv`, through 2020-08-21) is not written by this notebook.

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path("data")     # H&M CSVs
output_dir = Path(".")      # weekly files are written here

trans = pd.read_csv(data_dir / "transactions.csv")
trans["parsed_date"] = pd.to_datetime(trans["t_dat"])

In [ ]:
first_date = trans["parsed_date"].min()
last_date = trans["parsed_date"].max()
print(f"{len(trans):,} transactions from {first_date.date()} to {last_date.date()}")

In [ ]:
# The last month of transactions goes into the weekly files
cutoff_date = last_date - pd.DateOffset(months=1)
print(f"Cutoff: {cutoff_date.date()} (weekly files start the day after)")

In [ ]:
recent = trans[trans["parsed_date"] > cutoff_date]
iso = recent["parsed_date"].dt.isocalendar()

# One file per ISO week (Monday to Sunday), named by the first and last dates it contains
for (year, week), group in recent.groupby([iso["year"], iso["week"]]):
    first_day = group["parsed_date"].min().date()
    last_day = group["parsed_date"].max().date()

    filename = output_dir / f"week_{week:02d}_{first_day}_to_{last_day}.csv"
    group.to_csv(filename, index=False)
    print(f"Saved {len(group):,} rows to {filename.name}")